# eval_external_2023 — frozen pipeline, one-shot external test (TREC 2023 CT)

Applies the **exact** headline pipeline (frozen `R = elig_first-L512`, multi-view + NQS) to the
**blind TREC 2023** corpus, once. NDCG@10 is the generalization number; the 2023 qrels touch only
the final `pytrec_metrics` call — not the pool, the LLM floor, NQS, or any tuning.

**Resumable:** every stage caches to `trec2023/*.jsonl` on Drive and resumes per-topic; the corpus
dense-encode is chunk-checkpointed. A Colab disconnect never restarts a completed stage — just re-run
all cells and it picks up where it left off. Qwen is loaded lazily (skipped entirely on a cached resume).

**PREREQUISITES — verify on Drive before spending GPU:**
1. `build_corpus_2023.ipynb` re-run → `trec2023/{doc_fulltext_2023.jsonl (per-field), index2docid_2023.txt, topics2023_text.jsonl, qrels2023.txt}`.
2. `train_ensemble_full.ipynb` (POOL_TAG='nqs') re-run → `models/ensemble_nqs.txt` + `models/ensemble_nqs_features.json` (the ensemble behind the 0.5750 TREC22 headline).
3. Component checkpoints reachable via `resolve_ckpt`: clf_R, clf_topic, retriever-v2, Qwen judge, SapBERT.

**One-shot discipline:** decide nothing from the 2023 number; run once, report it. **Confound to state:** 2023 topics are questionnaire-format (domain shift) — a generalization stress test, not a like-for-like TREC22 rerun.

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q lightgbm pytrec_eval rank-bm25 sentence-transformers transformers accelerate datasets tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json
os.environ['HF_HUB_DISABLE_XET'] = '1'; os.environ['HF_HOME'] = '/content/hf_cache'
import numpy as np, torch, lightgbm as lgb
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from ctmatch.experiments import (ExperimentConfig, build_bm25, retrieval_blob, rrf_fuse,
    llm_expand_query, cross_encoder_scores, relevant_index, resolve_ckpt,
    llm_yesno_scores, llm_prompt, llm_topicality_prompt, topicality_blob,
    pytrec_metrics, write_trec_run)
DATA_ROOT = '/content/drive/MyDrive/ct_data23'
T23 = f'{DATA_ROOT}/trec2023'
cfg = ExperimentConfig(data_root=DATA_ROOT, pool_tag='nqs')   # frozen R + headline pool
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# The persisted ensemble IS the model behind the TREC22 headline — loaded, never refit here.
booster = lgb.Booster(model_file=cfg.path('models/ensemble_nqs.txt'))
FEAT_ORDER = json.load(open(cfg.path('models/ensemble_nqs_features.json')))
assert len(FEAT_ORDER) == booster.num_feature(), \
    f'feature-count mismatch: FEAT_ORDER={len(FEAT_ORDER)} vs booster={booster.num_feature()}'
print('repr:', cfg.repr_tag(), '| features:', FEAT_ORDER)

# Cache paths — every stage resumes from these.
P_EMB   = f'{T23}/doc_emb_2023_{cfg.retriever_ckpt.split("/")[-1]}.npy'
P_BM25  = f'{T23}/bm25_2023.pkl'
P_EXP   = f'{T23}/nqs_expansions_2023.jsonl'
P_POOL  = f'{T23}/pool_nqs_2023.json'
P_RFEAT = f'{T23}/retrieval_feats_2023.jsonl'
P_LLM   = f'{T23}/llm_scores_2023.jsonl'
P_TOPI  = f'{T23}/topicality_2023.jsonl'
P_CLFR  = f'{T23}/ce_clf_R_2023.jsonl'
P_CLFT  = f'{T23}/ce_clf_topic_2023.jsonl'
P_CM    = f'{T23}/condition_match_exp_2023.jsonl'
P_RUN   = f'{T23}/run_ext2023.txt'
P_METS  = f'{T23}/metrics_ext2023.json'

# Lazy Qwen — shared by NQS expansion + both judges; never loaded on a fully-cached resume.
_QWEN = {}
def get_qwen():
    if 'm' not in _QWEN:
        from transformers import AutoTokenizer, AutoModelForCausalLM
        tk = AutoTokenizer.from_pretrained(resolve_ckpt(cfg, cfg.llm_ckpt), padding_side='left')
        if tk.pad_token is None: tk.pad_token = tk.eos_token
        _QWEN['t'] = tk
        _QWEN['m'] = AutoModelForCausalLM.from_pretrained(resolve_ckpt(cfg, cfg.llm_ckpt),
                        torch_dtype=torch.float16, device_map='auto').eval()
    return _QWEN['m'], _QWEN['t']
def free_qwen():
    if 'm' in _QWEN:
        del _QWEN['m']; _QWEN.clear(); torch.cuda.empty_cache()

def load_feat(path, valkeys):
    """Per-(topic,doc) feature cache -> ({(t,d): val|tuple}, set_of_done_topics)."""
    d, done = {}, set()
    if os.path.exists(path):
        for l in open(path):
            r = json.loads(l); d[(r['topic_id'], r['doc_id'])] = (
                tuple(r[x] for x in valkeys) if len(valkeys) > 1 else r[valkeys[0]])
            done.add(r['topic_id'])
    return d, done

In [ ]:
# Load the 2023 corpus (per-field records) + topics + qrels directly. Everything downstream uses the
# corpus-agnostic ctmatch.experiments representation/scoring functions, so the representation is
# IDENTICAL to the TREC22 headline pipeline.
corpus_ids = [l.strip() for l in open(f'{T23}/index2docid_2023.txt') if l.strip()]
id2fields = {}
for l in open(f'{T23}/doc_fulltext_2023.jsonl'):
    r = json.loads(l); id2fields[r.get('nct_id') or r.get('doc_id')] = r
corpus_fields = [id2fields.get(d, {}) for d in corpus_ids]
topics = {r['topic_id']: r['topic_text'] for r in map(json.loads, open(f'{T23}/topics2023_text.jsonl'))}
rel = {}
for l in open(f'{T23}/qrels2023.txt'):
    t, _, d, r = l.split(); rel.setdefault(t, {})[d] = int(r)
topics = {t: x for t, x in topics.items() if t in rel}   # judged topics only
print(f'2023 corpus {len(corpus_ids):,} | judged topics {len(topics)}')

In [ ]:
# BM25 (cache) + dense encode (CHUNK-CHECKPOINTED so a disconnect resumes mid-encode).
import pickle
if os.path.exists(P_BM25):
    bm25 = pickle.load(open(P_BM25, 'rb'))
else:
    bm25 = build_bm25(corpus_fields, cfg); pickle.dump(bm25, open(P_BM25, 'wb'))

if os.path.exists(P_EMB):
    doc_emb = np.load(P_EMB)
else:
    enc = SentenceTransformer(resolve_ckpt(cfg, cfg.retriever_ckpt)); enc.max_seq_length = cfg.retriever_max_tokens
    blobs = [retrieval_blob(f, cfg) for f in corpus_fields]
    part, marker, CHUNK = P_EMB + '.part.npy', P_EMB + '.done', 50000
    if os.path.exists(part) and os.path.exists(marker):
        start = int(open(marker).read()); acc = [np.load(part)]; print(f'resuming encode from {start:,}')
    else:
        start, acc = 0, []
    for i in range(start, len(blobs), CHUNK):
        acc.append(enc.encode(blobs[i:i+CHUNK], convert_to_numpy=True, normalize_embeddings=True,
                              batch_size=256, show_progress_bar=True).astype(np.float32))
        np.save(part, np.vstack(acc)); open(marker, 'w').write(str(i + CHUNK))
    doc_emb = np.vstack(acc); np.save(P_EMB, doc_emb)
    for p in (part, marker):
        if os.path.exists(p): os.remove(p)
    del enc; torch.cuda.empty_cache()
q_enc = SentenceTransformer(resolve_ckpt(cfg, cfg.retriever_ckpt)); q_enc.max_seq_length = cfg.retriever_max_tokens
print('retrieval ready', doc_emb.shape)

In [ ]:
# NQS expansion per topic (cached; Qwen loaded only if there is work to do).
expansion, done = {}, set()
if os.path.exists(P_EXP):
    for l in open(P_EXP):
        r = json.loads(l); expansion[r['topic_id']] = r['expansion']; done.add(r['topic_id'])
todo = [t for t in topics if t not in done]
if todo:
    qwen, qtok = get_qwen()
    with open(P_EXP, 'a') as f:
        for t in tqdm(todo, desc='nqs expand'):
            e = llm_expand_query(qwen, qtok, topics[t], cfg); expansion[t] = e
            f.write(json.dumps({'topic_id': t, 'expansion': e}) + '\n'); f.flush()
print('expansions:', len(expansion))

In [ ]:
# NQS candidate pool + retrieval features (mirrors nqs_retrieval.retrieve; NQS query = topic + ' ' + expansion).
def retrieve(qtext, k=cfg.cand_k):
    sc = bm25.get_scores(qtext.lower().split()); bt = np.argpartition(-sc, k)[:k]; bt = bt[np.argsort(-sc[bt])]
    qv = q_enc.encode([qtext], normalize_embeddings=True)[0].astype('float32'); sims = doc_emb @ qv
    dt = np.argpartition(-sims, k)[:k]; dt = dt[np.argsort(-sims[dt])]
    bm = {corpus_ids[i]: float(sc[i]) for i in bt}; dn = {corpus_ids[i]: float(sims[i]) for i in dt}
    br = {d: r for r, d in enumerate(bm)}; dr = {d: r for r, d in enumerate(dn)}
    rrf = rrf_fuse([list(br), list(dr)], k=cfg.rrf_k)
    cand = sorted(set(bm) | set(dn), key=lambda d: rrf.get(d, 0), reverse=True)
    feats = {d: {'bm25': bm.get(d, 0.), 'dense': dn.get(d, 0.), 'rrf': rrf.get(d, 0.),
                 'bm25_rank': br.get(d, k), 'dense_rank': dr.get(d, k)} for d in cand}
    return cand, feats

if os.path.exists(P_POOL) and os.path.exists(P_RFEAT):
    pool = json.load(open(P_POOL)); rfeat = {}
    for l in open(P_RFEAT):
        r = json.loads(l); rfeat[(r['topic_id'], r['doc_id'])] = {k: r[k] for k in ('bm25','dense','rrf','bm25_rank','dense_rank')}
else:
    pool, rfeat = {}, {}
    with open(P_RFEAT, 'w') as f:
        for t in tqdm(topics, desc='nqs retrieve'):
            cand, feats = retrieve(topics[t] + ' ' + expansion[t])
            pool[t] = cand
            for d in cand:
                rfeat[(t, d)] = feats[d]
                f.write(json.dumps({'topic_id': t, 'doc_id': d, **feats[d]}) + '\n')
    json.dump(pool, open(P_POOL, 'w'))
print('pool | mean cand/topic', int(np.mean([len(v) for v in pool.values()])))

In [ ]:
# Both LLM judges over the top-500 by RRF (docs[:llm_top_k]) — same slice the feature notebooks use.
# Cached + per-topic resumable; Qwen freed once judges are done.
llm_yesno, done_e = load_feat(P_LLM, ['llm_score'])
topicality, done_t = load_feat(P_TOPI, ['topicality'])
todo_e = [t for t in topics if t not in done_e]
todo_t = [t for t in topics if t not in done_t]
if todo_e or todo_t:
    qwen, qtok = get_qwen()
    if todo_e:
        with open(P_LLM, 'a') as f:
            for t in tqdm(todo_e, desc='elig judge'):
                top = [d for d in pool[t][:cfg.llm_top_k] if d in id2fields]; ff = [id2fields[d] for d in top]
                for d, s in zip(top, llm_yesno_scores(qwen, qtok, topics[t], ff, cfg, batch=8, prompt_fn=llm_prompt)):
                    llm_yesno[(t, d)] = s
                    f.write(json.dumps({'topic_id': t, 'doc_id': d, 'llm_score': float(s)}) + '\n')
                f.flush()
    if todo_t:
        with open(P_TOPI, 'a') as f:
            for t in tqdm(todo_t, desc='topicality judge'):
                top = [d for d in pool[t][:cfg.llm_top_k] if d in id2fields]; ff = [id2fields[d] for d in top]
                for d, s in zip(top, llm_yesno_scores(qwen, qtok, topics[t], ff, cfg, batch=8, prompt_fn=llm_topicality_prompt)):
                    topicality[(t, d)] = s
                    f.write(json.dumps({'topic_id': t, 'doc_id': d, 'topicality': float(s)}) + '\n')
                f.flush()
free_qwen()
print('judges done')

In [ ]:
# Cross-encoder views over the WHOLE pool: clf_R (elig_first) + clf_topic (topic_first).
# Cached + per-topic resumable; each model loaded then freed.
from transformers import AutoTokenizer as AT, AutoModelForSequenceClassification as ASC
def ce_stage(path, ckpt, ce_cfg, desc):
    R, P, done = {}, {}, set()
    if os.path.exists(path):
        for l in open(path):
            r = json.loads(l); R[(r['topic_id'], r['doc_id'])] = r['rel']; P[(r['topic_id'], r['doc_id'])] = r['partial']; done.add(r['topic_id'])
    todo = [t for t in topics if t not in done]
    if todo:
        rk = resolve_ckpt(cfg, ckpt); tk = AT.from_pretrained(rk); m = ASC.from_pretrained(rk).to(device).eval()
        ridx = relevant_index(m); pidx = next((int(i) for i, v in m.config.id2label.items() if 'partial' in str(v).lower()), None)
        with open(path, 'a') as f:
            for t in tqdm(todo, desc=desc):
                docs = [d for d in pool[t] if d in id2fields]; ff = [id2fields[d] for d in docs]
                rr = cross_encoder_scores(m, tk, topics[t], ff, ce_cfg, ridx)
                pp = cross_encoder_scores(m, tk, topics[t], ff, ce_cfg, pidx) if pidx is not None else [0.] * len(docs)
                for d, a, b in zip(docs, rr, pp):
                    R[(t, d)] = a; P[(t, d)] = b
                    f.write(json.dumps({'topic_id': t, 'doc_id': d, 'rel': a, 'partial': b}) + '\n')
                f.flush()
        del m; torch.cuda.empty_cache()
    return R, P
clf_rel, clf_partial = ce_stage(P_CLFR, cfg.clf_ckpt, cfg, 'clf_R')
clf_topic_rel, clf_topic_partial = ce_stage(P_CLFT, cfg.clf_topic_ckpt, cfg.with_(repr_strategy='topic_first'), 'clf_topic')
print('cross-encoders done')

In [ ]:
# condition_match_exp: SapBERT cosine(expanded query, doc topicality blob) over the whole pool. Cached.
condition_match, done_cm = load_feat(P_CM, ['condition_match'])
todo = [t for t in topics if t not in done_cm]
if todo:
    sap = SentenceTransformer(resolve_ckpt(cfg, cfg.topicality_encoder))
    uniq = sorted({d for docs in pool.values() for d in docs if d in id2fields})
    demb = sap.encode([topicality_blob(id2fields[d], cfg) for d in uniq],
                      normalize_embeddings=True, batch_size=128, show_progress_bar=True).astype('float32')
    didx = {d: i for i, d in enumerate(uniq)}
    qv = sap.encode([topics[t] + '. ' + expansion.get(t, '') for t in todo],   # expanded query (cm_exp)
                    normalize_embeddings=True, batch_size=128).astype('float32')
    with open(P_CM, 'a') as f:
        for t, v in zip(todo, qv):
            for d in pool[t]:
                if d in didx:
                    s = float(v @ demb[didx[d]]); condition_match[(t, d)] = s
                    f.write(json.dumps({'topic_id': t, 'doc_id': d, 'condition_match': s}) + '\n')
            f.flush()
    del sap; torch.cuda.empty_cache()
print('condition_match_exp done')

In [ ]:
# Assemble feature vectors in the PERSISTED column order, predict once, save the run (durable).
def featvec(t, d):
    rf = rfeat.get((t, d), {})
    v = {'bm25': rf.get('bm25', 0.), 'bm25_rank': rf.get('bm25_rank', cfg.cand_k),
         'dense': rf.get('dense', 0.), 'dense_rank': rf.get('dense_rank', cfg.cand_k), 'rrf': rf.get('rrf', 0.),
         'clf_rel': clf_rel.get((t, d), 0.), 'clf_partial': clf_partial.get((t, d), 0.),
         'clf_topic_rel': clf_topic_rel.get((t, d), 0.), 'clf_topic_partial': clf_topic_partial.get((t, d), 0.),
         'llm_yesno': llm_yesno.get((t, d), cfg.llm_floor),
         'topicality': topicality.get((t, d), 0.),
         'condition_match': condition_match.get((t, d), 0.)}
    return [v[f] for f in FEAT_ORDER]
run = {}
for t in topics:
    docs = [d for d in pool[t] if d in id2fields]
    X = np.array([featvec(t, d) for d in docs], dtype=np.float32)
    run[t] = {d: float(s) for d, s in zip(docs, booster.predict(X))}
write_trec_run(P_RUN, run, 'ctmatch_ext2023')
print('scored', len(run), 'topics ->', P_RUN)

In [ ]:
# One-shot metrics — qrels used ONLY here. NDCG@10 graded; P@10/MRR eligible-only (TREC overview basis).
import pytrec_eval
qrels = {t: {d: int(r) for d, r in rel[t].items()} for t in run}
metrics = pytrec_metrics(run, qrels, k=10)
per = pytrec_eval.RelevanceEvaluator(qrels, {'ndcg_cut.10'}).evaluate(run)
vals = np.array([per[t]['ndcg_cut_10'] for t in per])
boot = [np.mean(np.random.default_rng(i).choice(vals, len(vals), replace=True)) for i in range(10000)]
ci = [round(float(np.percentile(boot, 2.5)), 4), round(float(np.percentile(boot, 97.5)), 4)]
metrics['ndcg@10_ci'] = ci
json.dump(metrics, open(P_METS, 'w'))
print('=== TREC 2023 (external, frozen pipeline) — ONE-SHOT ===')
print(metrics)
print(f'NDCG@10 = {vals.mean():.4f}  95% CI {ci}  (n={len(vals)})')
print('Reference: TREC22 headline 0.5750. Fill TREC23 best-run NDCG@10 from the 2023 CT overview for context.')